# Segmasker Low-Complexity Detection Example

This notebook demonstrates how to detect **low-complexity regions** in protein sequences using **NCBI Segmasker**.

Low-complexity regions (e.g., poly-alanine, proline-rich stretches) can indicate disordered regions, cause false positives in homology searches, or flag problematic protein designs.

## Imports

In [1]:
from proto_tools import SegmaskerConfig, SegmaskerInput, run_segmasker


## 1. Detect Low-Complexity Regions

Scan protein sequences for low-complexity segments using the SEG algorithm.

### API Reference

**`SegmaskerInput`**

| Field | Type | Description |
|-------|------|-------------|
| `sequences` | `List[str]` | Protein sequence(s) to analyze |

**`SegmaskerConfig`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `window` | `int` | `15` | Sliding window size |
| `locut` | `float` | `1.8` | Low-complexity threshold |
| `hicut` | `float` | `3.4` | High-complexity threshold |

**`SegmaskerOutput`**

| Field | Type | Description |
|-------|------|-------------|
| `low_complexity_fractions` | `List[float]` | Fraction of low-complexity residues per sequence (0.0-1.0) |
| `low_complexity_counts` | `List[int]` | Count of low-complexity residues per sequence |
| `sequence_lengths` | `List[int]` | Length of each input sequence |
| `results_df` | `Optional[DataFrame]` | Detailed per-region results |

In [2]:
# Compare a well-folded protein vs one with a low-complexity stretch
sequences = [
    "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH",  # Hemoglobin alpha (well-folded)
    "MVLSPADKAAAAAAAAAAAAAAAAAAAAAAGAEALERMFLSFPTTKTYFPHF",  # Artificial poly-alanine insert
]

inputs = SegmaskerInput(sequences=sequences)
config = SegmaskerConfig()

result = run_segmasker(inputs, config)

labels = ["Hemoglobin alpha", "Poly-Ala insert"]
for label, frac, count, length in zip(
    labels, result.low_complexity_fractions, result.low_complexity_counts, result.sequence_lengths
):
    print(f"{label}: {frac:.1%} low-complexity ({count}/{length} residues)")

Hemoglobin alpha: 0.0% low-complexity (0/51 residues)
Poly-Ala insert: 50.0% low-complexity (26/52 residues)


## 2. Export Results

Save results to CSV or JSON for downstream filtering.

In [3]:
from pathlib import Path

output_dir = Path("./example_output")
result.export(name="segmasker_results", export_path=output_dir, file_format="csv")
print(f"Segmasker results exported to {output_dir}")

Segmasker results exported to example_output
